# 06 — Deep Validation
## Designed to BREAK the model, not confirm it

Every test here is adversarial. We are trying to find the flaw.

| Test | What it checks | Red flag |
|------|---------------|----------|
| 1. Label shuffle | Future leakage in features | Accuracy stays high with random labels |
| 2. Permutation | Features doing real work | Accuracy stays high with shuffled rows |
| 3. Walk-forward decay | Temporal generalization | No decay as we move from training |
| 4. Candle alignment | No look-ahead bias | Features use future candle data |
| 5. Lag test | Feature persistence | Accuracy unchanged when features shifted +1 |
| 6. Random feature baseline | Architecture memorization | >55% accuracy with pure noise features |

In [ ]:
# ── Imports & Setup ────────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

DARK  = '#080c14'
BLUE  = '#4fc3f7'
GOLD  = '#ffd700'
RED   = '#ef5350'
GREEN = '#66bb6a'

DATA_DIR   = Path('../backend/data/features')
MODELS_DIR = Path('../backend/models')

HORIZONS       = ['1H', '4H', '1D', '7D']
QUANTILES      = [0.10, 0.25, 0.50, 0.75, 0.90]
QUANTILE_NAMES = ['Q10', 'Q25', 'Q50', 'Q75', 'Q90']
PAIRS          = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF', 'AUDUSD', 'USDCAD', 'NZDUSD']

# ── Load data ──────────────────────────────────────────────────
print('Loading features...')
df = pd.read_parquet(DATA_DIR / 'all_pairs_features_labels.parquet')
df.index = pd.to_datetime(df.index)
print(f'Loaded: {len(df):,} rows | {df.index.min().date()} → {df.index.max().date()}')

feature_cols = [c for c in df.columns if not c.startswith('label_') and c != 'pair']
print(f'Features: {len(feature_cols)}')

# ── Load official LumenY models ────────────────────────────────
print('Loading official models...')
models = {}
calibrators = {}
for horizon in HORIZONS:
    calibrators[horizon] = joblib.load(MODELS_DIR / f'calibrator_{horizon}.joblib')
    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        bundle = joblib.load(MODELS_DIR / f'model_{horizon}_Q{int(q*100)}.joblib')
        models[(horizon, q_name)] = bundle['model']
print('Models loaded.')

# ── Splits (same as official training) ────────────────────────
df_train = df[df.index < '2020-01-01']
df_cal   = df[(df.index >= '2020-01-01') & (df.index < '2022-01-01')]
df_test  = df[df.index >= '2022-01-01']

# EURUSD only, non-overlapping for clean tests
df_eu_test = df_test[df_test['pair'] == 'EURUSD']

print(f'Train: {len(df_train):,} | Cal: {len(df_cal):,} | Test: {len(df_test):,}')

In [ ]:
# ── Shared helpers ─────────────────────────────────────────────

def derive_p_down(q_vals):
    qs = np.array(QUANTILES)
    vals = np.sort(q_vals)
    if vals[0] <= 0 <= vals[-1]:
        return float(np.interp(0, vals, qs))
    elif vals[-1] < 0:
        slope = (qs[-1] - qs[-2]) / (vals[-1] - vals[-2] + 1e-10)
        return float(np.clip(qs[-1] + slope * (0 - vals[-1]), 0.90, 0.999))
    else:
        slope = (qs[1] - qs[0]) / (vals[1] - vals[0] + 1e-10)
        return float(np.clip(qs[0] + slope * (0 - vals[0]), 0.001, 0.10))


def get_predictions_official(df_subset, horizon):
    """Run official LumenY models on a subset."""
    X = df_subset[feature_cols].ffill()
    y = df_subset[f'label_{horizon}'].values

    q_preds = {}
    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        q_preds[q_name] = models[(horizon, q_name)].predict(X)

    p_raw = np.array([
        derive_p_down(np.array([q_preds[n][i] for n in QUANTILE_NAMES]))
        for i in range(len(X))
    ])

    p_cal = calibrators[horizon].predict(p_raw)
    p_dom = np.where(p_cal > 0.5, p_cal, 1 - p_cal)
    correct = np.where(p_cal > 0.5, y < 0, y > 0)
    return p_cal, p_dom, correct, y


def quick_lgbm(X_tr, y_tr, X_val, y_val, X_te, quantile=0.50, n_est=500):
    """Train a single quick LightGBM model for validation tests."""
    model = lgb.LGBMRegressor(
        objective='quantile', alpha=quantile,
        n_estimators=n_est, learning_rate=0.05,
        num_leaves=63, device='gpu', verbosity=-1
    )
    model.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(30, verbose=False)])
    return model.predict(X_te), model


print('Helpers ready.')

---
## TEST 1 — Label Shuffle Test

**What it checks:** Future leakage in features.

**Method:** Randomly shuffle the labels (break the time relationship between features and outcomes), retrain a fresh model on shuffled labels, evaluate.

**Expected result if model is clean:** ~50% accuracy — shuffled labels = pure noise, model learns nothing.

**Red flag:** If accuracy stays above 60%, the features contain future information that leaks regardless of label order.

In [ ]:
# TEST 1 — Label Shuffle
# We use Q50 only (median model) for speed. 3 runs to check variance.

HORIZON = '1H'
N_RUNS  = 3

df_eu_train = df_train[df_train['pair'] == 'EURUSD']
df_eu_cal   = df_cal[df_cal['pair'] == 'EURUSD']

X_train = df_eu_train[feature_cols].ffill().values
X_cal   = df_eu_cal[feature_cols].ffill().values
X_test  = df_eu_test[feature_cols].ffill().values
y_train_real = df_eu_train[f'label_{HORIZON}'].values
y_cal_real   = df_eu_cal[f'label_{HORIZON}'].values
y_test_real  = df_eu_test[f'label_{HORIZON}'].values

# Real model accuracy (baseline)
_, p_dom_real, correct_real, _ = get_predictions_official(df_eu_test, HORIZON)
real_acc = correct_real.mean()
real_acc_70 = correct_real[p_dom_real >= 0.70].mean()

print(f'TEST 1 — Label Shuffle | Horizon: {HORIZON}')
print('=' * 55)
print(f'Real model accuracy:      {real_acc:.1%}')
print(f'Real model ≥70% accuracy: {real_acc_70:.1%}')
print(f'\nShuffled label runs (expected ~50%):')  
print(f'{"Run":<6} {"Overall acc":<15} {"Notes"}')
print('-' * 40)

shuffle_accs = []
for run in range(N_RUNS):
    np.random.seed(run * 42)
    
    # Shuffle labels — break time relationship
    y_train_shuffled = np.random.permutation(y_train_real)
    y_cal_shuffled   = np.random.permutation(y_cal_real)
    
    preds, _ = quick_lgbm(X_train, y_train_shuffled,
                           X_cal,   y_cal_shuffled,
                           X_test)
    
    # Derive direction from Q50 prediction
    pred_dir_down = preds < 0
    actual_dir_down = y_test_real < 0
    acc = (pred_dir_down == actual_dir_down).mean()
    shuffle_accs.append(acc)
    
    flag = '✓ OK' if acc < 0.60 else '⚠ SUSPICIOUS'
    print(f'{run+1:<6} {acc:<15.1%} {flag}')

print(f'\nMean shuffled accuracy: {np.mean(shuffle_accs):.1%}')
print(f'Real model accuracy:    {real_acc:.1%}')
print(f'Gap:                    {real_acc - np.mean(shuffle_accs):+.1%}')

if np.mean(shuffle_accs) < 0.55:
    print('\n✓ CLEAN — Shuffled labels → ~50% accuracy. No future leakage detected.')
else:
    print('\n⚠ WARNING — Shuffled labels give >55% accuracy. Investigate leakage.')

---
## TEST 2 — Row Permutation Test

**What it checks:** Whether features are doing real work.

**Method:** Keep labels intact but shuffle the feature rows randomly (break the time relationship from the feature side). Apply official models to shuffled features.

**Expected result if model is real:** Accuracy drops to ~50% — shuffled features destroy the signal.

**Red flag:** Accuracy stays high — means the model doesn't actually need the correct features, suggesting something structural is wrong.

In [ ]:
# TEST 2 — Row Permutation
# Shuffle feature rows, keep labels in original order, run official models

print('TEST 2 — Row Permutation')
print('=' * 55)

N_RUNS = 3

for horizon in ['1H', '4H', '1D']:
    _, p_dom_real, correct_real, y_real = get_predictions_official(df_eu_test, horizon)
    real_acc = correct_real.mean()

    perm_accs = []
    for run in range(N_RUNS):
        np.random.seed(run * 7)
        
        # Shuffle rows of test features
        df_shuffled = df_eu_test.copy()
        shuffled_idx = np.random.permutation(len(df_shuffled))
        df_shuffled[feature_cols] = df_eu_test[feature_cols].values[shuffled_idx]
        
        _, _, correct_perm, _ = get_predictions_official(df_shuffled, horizon)
        perm_accs.append(correct_perm.mean())
    
    mean_perm = np.mean(perm_accs)
    flag = '✓ OK' if mean_perm < 0.55 else '⚠ SUSPICIOUS'
    print(f'{horizon}: real={real_acc:.1%}  permuted={mean_perm:.1%}  drop={real_acc-mean_perm:+.1%}  {flag}')

print('\nExpected: permuted accuracy ~50% (features carry all the signal)')

---
## TEST 3 — Walk-Forward Decay

**What it checks:** Whether accuracy degrades as we move further from training data.

**Method:** Split the test set into quarterly windows. Measure accuracy in each quarter.

**Expected result if model is real:** Gradual, modest decay over time as market evolves.

**Red flag:** Accuracy perfectly flat or INCREASING over time — too good to be true, suggests leakage.

In [ ]:
# TEST 3 — Walk-Forward Decay

print('TEST 3 — Walk-Forward Decay (Quarterly)')
print('=' * 65)

quarters = pd.period_range('2022Q1', '2025Q4', freq='Q')

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.patch.set_facecolor(DARK)

for idx, horizon in enumerate(HORIZONS):
    ax = axes[idx // 2][idx % 2]
    ax.set_facecolor(DARK)

    q_accs, q_freqs, q_labels = [], [], []

    for q in quarters:
        start = q.start_time
        end   = q.end_time
        mask  = (df_eu_test.index >= start) & (df_eu_test.index <= end)
        df_q  = df_eu_test[mask]
        
        if len(df_q) < 100:
            continue
        
        _, p_dom, correct, _ = get_predictions_official(df_q, horizon)
        
        mask_70 = p_dom >= 0.70
        if mask_70.sum() < 20:
            continue
        
        acc_70 = correct[mask_70].mean()
        freq_70 = mask_70.mean()
        q_accs.append(acc_70)
        q_freqs.append(freq_70)
        q_labels.append(str(q))

    if not q_accs:
        continue

    x = range(len(q_labels))
    ax.bar(x, q_accs, color=BLUE, alpha=0.7)
    ax.axhline(np.mean(q_accs), color=GOLD, linewidth=1.5, linestyle='--')
    ax.axhline(0.50, color='white', linewidth=0.5, linestyle='--', alpha=0.3)
    ax.set_xticks(x)
    ax.set_xticklabels(q_labels, rotation=45, fontsize=7, color='white')
    ax.set_ylim(0.5, 1.0)
    ax.set_title(f'{horizon} ≥70% Accuracy by Quarter | Mean={np.mean(q_accs):.1%}  Std={np.std(q_accs):.3f}',
                 color='white', fontsize=9)
    ax.tick_params(colors='white', labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor('#1a2332')

    # Trend
    if len(q_accs) > 3:
        z = np.polyfit(range(len(q_accs)), q_accs, 1)
        trend = z[0] * len(q_accs)  # total change
        direction = 'degrading' if trend < -0.02 else 'stable' if abs(trend) < 0.02 else 'improving'
        print(f'{horizon}: mean={np.mean(q_accs):.1%}  std={np.std(q_accs):.3f}  trend={trend:+.3f} ({direction})')

plt.suptitle('Walk-Forward Decay — ≥70% Accuracy by Quarter\nGold = mean | Decay expected for real model',
             color='white', fontsize=11)
plt.tight_layout()
plt.show()

---
## TEST 4 — Candle Alignment Audit

**What it checks:** Look-ahead bias — are features accidentally using future candle data?

**Method:** For each timeframe, verify that the feature values at time T only use candle data from T and earlier. We do this by checking the maximum lag in each feature and comparing it to the label horizon.

**Red flag:** Any feature that uses data from candles that haven't closed at prediction time.

In [ ]:
# TEST 4 — Candle Alignment Audit
# Check if features at time T correlate with FUTURE returns more than PAST returns
# A clean model: features correlate with future (that's the point)
# A leaking model: features correlate with CONTEMPORANEOUS future in a suspicious way

print('TEST 4 — Candle Alignment Audit')
print('=' * 65)
print('Checking correlation of features with future vs past returns...')
print()

df_eu = df[df['pair'] == 'EURUSD'].copy()
X_eu  = df_eu[feature_cols].ffill()

# For each horizon, check:
# 1. Correlation of label_H with features at time T (legitimate — this is what we train on)
# 2. Correlation of label_H with features at time T+H (suspicious — features from AFTER the label period)

for horizon, h_bars in [('1H', 1), ('4H', 4), ('1D', 24)]:
    y = df_eu[f'label_{horizon}'].values
    
    # Sample top 10 most important features for this horizon
    bundle = joblib.load(MODELS_DIR / f'model_{horizon}_Q50.joblib')
    imp = bundle['model'].feature_importances_
    top_idx = np.argsort(imp)[-10:]
    top_features = [feature_cols[i] for i in top_idx]
    
    print(f'Horizon: {horizon}')
    print(f'  {"Feature":<30} {"Corr(feat_t, label_t)":>22} {"Corr(feat_t+H, label_t)":>24} {"Suspicious?"}')
    print(f'  {"-"*82}')
    
    suspicious = []
    for feat in top_features:
        x_t   = X_eu[feat].values
        x_fut = X_eu[feat].shift(-h_bars).values  # features from FUTURE
        
        # Correlation at time T (normal)
        valid = ~(np.isnan(x_t) | np.isnan(y))
        corr_t = np.corrcoef(x_t[valid], y[valid])[0,1] if valid.sum() > 100 else float('nan')
        
        # Correlation with future features (suspicious if much higher)
        valid_f = ~(np.isnan(x_fut) | np.isnan(y))
        corr_fut = np.corrcoef(x_fut[valid_f], y[valid_f])[0,1] if valid_f.sum() > 100 else float('nan')
        
        # If future correlation >> current correlation, suspicious
        if not np.isnan(corr_t) and not np.isnan(corr_fut):
            ratio = abs(corr_fut) / (abs(corr_t) + 1e-10)
            flag = '⚠ SUSPICIOUS' if ratio > 3.0 and abs(corr_fut) > 0.1 else '✓'
            if '⚠' in flag:
                suspicious.append(feat)
        else:
            flag = '?'
        
        print(f'  {feat:<30} {corr_t:>22.4f} {corr_fut:>24.4f} {flag}')
    
    if suspicious:
        print(f'\n  ⚠ SUSPICIOUS features: {suspicious}')
    else:
        print(f'\n  ✓ No suspicious alignment detected for {horizon}')
    print()

---
## TEST 5 — Lag Test

**What it checks:** Feature persistence — does the model still work if features are shifted forward by 1 candle?

**Method:** Shift all feature rows forward by 1 bar (use features from T-1 to predict label at T). Run official models.

**Expected result if model is real:** Accuracy should drop meaningfully — using yesterday's features to predict today should be worse than using today's features.

**Red flag:** Accuracy barely changes — means features are so persistent/slow-moving that 1-bar lag doesn't matter, suggesting they may be carrying forward-looking information.

In [ ]:
# TEST 5 — Lag Test

print('TEST 5 — Lag Test')
print('=' * 65)
print('Shifting features forward by N bars and measuring accuracy drop')
print()

lags = [0, 1, 2, 4, 8, 24]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor(DARK)

for h_idx, horizon in enumerate(['1H', '4H', '1D']):
    ax = axes[h_idx]
    ax.set_facecolor(DARK)
    
    lag_accs = []
    
    for lag in lags:
        df_lagged = df_eu_test.copy()
        
        if lag == 0:
            pass  # no shift
        else:
            # Shift features forward by lag bars (use older features)
            df_lagged[feature_cols] = df_eu_test[feature_cols].shift(lag).values
        
        df_lagged = df_lagged.dropna(subset=[feature_cols[0]])
        _, p_dom, correct, _ = get_predictions_official(df_lagged, horizon)
        
        mask_70 = p_dom >= 0.70
        acc = correct[mask_70].mean() if mask_70.sum() > 20 else float('nan')
        lag_accs.append(acc)
        print(f'{horizon} lag={lag:>2}: acc={acc:.1%}')
    
    ax.plot(lags, lag_accs, 'o-', color=BLUE, linewidth=2, markersize=6)
    ax.axhline(0.5, color='white', linewidth=0.5, linestyle='--', alpha=0.3)
    ax.set_xlabel('Lag (bars)', color='white')
    ax.set_ylabel('≥70% Accuracy', color='white')
    ax.set_title(f'{horizon} — Accuracy vs Feature Lag', color='white', fontsize=10)
    ax.tick_params(colors='white')
    ax.set_ylim(0.45, 1.0)
    for spine in ax.spines.values():
        spine.set_edgecolor('#1a2332')
    print()

plt.suptitle('Lag Test — Accuracy Should Decay as Features Get Older\nSharp drop = features are time-sensitive (good)',
             color='white', fontsize=11)
plt.tight_layout()
plt.show()

---
## TEST 6 — Random Feature Baseline

**What it checks:** Architecture memorization — can the model get good results even with pure noise features?

**Method:** Replace all 208 features with random Gaussian noise. Same architecture, same labels, same splits. Train and evaluate.

**Expected result if architecture is clean:** ~50% accuracy — noise features → no signal → coin flip.

**Red flag:** Accuracy above 55% — the architecture itself is memorizing the label distribution.

In [ ]:
# TEST 6 — Random Feature Baseline

print('TEST 6 — Random Feature Baseline')
print('=' * 55)
print('Replacing all 208 features with Gaussian noise...')
print('Expected: ~50% accuracy (architecture not memorizing)')
print()

HORIZON = '1H'
N_RUNS  = 3

df_eu_train = df_train[df_train['pair'] == 'EURUSD']
df_eu_cal   = df_cal[df_cal['pair'] == 'EURUSD']

y_train = df_eu_train[f'label_{HORIZON}'].values
y_cal   = df_eu_cal[f'label_{HORIZON}'].values
y_test  = df_eu_test[f'label_{HORIZON}'].values

real_acc = (get_predictions_official(df_eu_test, HORIZON)[2]).mean()
print(f'Real model accuracy: {real_acc:.1%}')
print(f'\nRandom feature runs:')
print(f'{"Run":<6} {"Accuracy":<12} {"Status"}')
print('-' * 30)

rand_accs = []
for run in range(N_RUNS):
    np.random.seed(run * 13)
    
    # Pure random noise features — same shape as real features
    X_rand_train = np.random.randn(len(df_eu_train), len(feature_cols))
    X_rand_cal   = np.random.randn(len(df_eu_cal),   len(feature_cols))
    X_rand_test  = np.random.randn(len(df_eu_test),  len(feature_cols))
    
    preds, _ = quick_lgbm(X_rand_train, y_train,
                           X_rand_cal,   y_cal,
                           X_rand_test,  n_est=300)
    
    pred_down   = preds < 0
    actual_down = y_test < 0
    acc = (pred_down == actual_down).mean()
    rand_accs.append(acc)
    
    flag = '✓ OK' if acc < 0.55 else '⚠ SUSPICIOUS'
    print(f'{run+1:<6} {acc:<12.1%} {flag}')

print(f'\nMean random accuracy: {np.mean(rand_accs):.1%}')
print(f'Real model accuracy:  {real_acc:.1%}')
print(f'Gap:                  {real_acc - np.mean(rand_accs):+.1%}')

if np.mean(rand_accs) < 0.55:
    print('\n✓ CLEAN — Random features → ~50%. Architecture is not memorizing.')
else:
    print('\n⚠ WARNING — Architecture achieves >55% with random features. Investigate.')

---
## SUMMARY
Run this cell after all tests complete.

In [ ]:
# DEEP VALIDATION SUMMARY

print('=' * 65)
print('DEEP VALIDATION SUMMARY')
print('=' * 65)
print()
print('Fill in results after running each test:')
print()
print(f'{"Test":<35} {"Expected":<15} {"Result":<15} {"Verdict"}')
print('-' * 75)
print(f'{"1. Label shuffle":<35} {"~50%":<15} {"???":<15} {"???"}')
print(f'{"2. Row permutation":<35} {"~50%":<15} {"???":<15} {"???"}')
print(f'{"3. Walk-forward decay":<35} {"Gradual drop":<15} {"???":<15} {"???"}')
print(f'{"4. Candle alignment":<35} {"No flags":<15} {"???":<15} {"???"}')
print(f'{"5. Lag test":<35} {"Decay at lag>1":<15} {"???":<15} {"???"}')
print(f'{"6. Random features":<35} {"~50%":<15} {"???":<15} {"???"}')
print()
print('If all 6 tests pass: the edge is real beyond reasonable doubt.')
print('If any test fails: investigate that specific issue before proceeding.')